## Estudo de caso 2: Probalidade de inadimplencia

In [1]:
import keras 
import kagglehub as kh

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
import pandas as pd

In [2]:
# Criando uma sessão Spark

spark = SparkSession.builder \
    .appName("LendingClubAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.hadoop.hadoop.native.lib", "false") \
    .config("spark.hadoop.io.nativeio.nativeio.enabled", "false") \
    .getOrCreate()

In [3]:
spark

Baixando os dados

In [4]:
# Download latest version
path = kh.dataset_download("wordsforthewise/lending-club")

In [5]:
print(path)

C:\Users\mateu\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3


Lendo conj. de dados

In [6]:
data = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{path}\\accepted_2007_to_2018q4.csv\\accepted_2007_to_2018Q4.csv")

In [7]:
data.show(2)

+--------+---------+---------+-----------+---------------+----------+--------+-----------+-----+---------+---------+----------+--------------+----------+-------------------+--------+-----------+----------+--------------------+----+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+-----------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-------

> `Target` é charge-off, ou divida irrecuperavel, quando o credor desiste de cobrar o devedor que deixou de pagar há varios meses.
> * 1 quando há charge-off e 0 quando não há.

> Esse conjundo de dados possui 150 variáveis

In [8]:
len(data.columns)

151

In [9]:
data.count()

2260701

> 2260701 linhas e 151 colunas

In [10]:
data.groupBy("loan_status").count().orderBy("count", ascending=False).show()

+--------------------+-------+
|         loan_status|  count|
+--------------------+-------+
|          Fully Paid|1076751|
|             Current| 878317|
|         Charged Off| 268558|
|  Late (31-120 days)|  21467|
|     In Grace Period|   8436|
|   Late (16-30 days)|   4349|
|Does not meet the...|   1988|
|Does not meet the...|    761|
|             Default|     40|
|                NULL|     33|
|            Oct-2015|      1|
+--------------------+-------+



> Vamos usar uma problema de classificação binaria `Charge Off`(irrecuperaiveis) e `Fully Paid`(Totalmente pagos), os restantes não são do nosso interesse

In [11]:
data = data.where(
    f.col("loan_status").isin(["Fully Paid", "Charged Off"])
)
data.count()

1345309

In [12]:
data.groupBy("loan_status").count()\
    .select([
        f.col("loan_status"),
        f.round(f.col('count')/data.count(), 4).alias("percentage")
    ]).show()

+-----------+----------+
|loan_status|percentage|
+-----------+----------+
| Fully Paid|    0.8004|
|Charged Off|    0.1996|
+-----------+----------+



> Os dados estão desbalanceados com 80 para emprestimos pagos totalmente e 20 irrecuperaveis

Convetendo a coluna loan_status para binario

In [13]:
data = data.withColumn(
    "loan_status", f.when(f.col("loan_status") == "Fully Paid", 0).otherwise(1)
)

In [14]:
data.select("loan_status").distinct().show()

+-----------+
|loan_status|
+-----------+
|          1|
|          0|
+-----------+



Removendo dados faltantes colunas com > 30% de dados faltantes serão removidos

In [15]:
subset = data.select(
    [(
        f.sum(f.when(f.col(c).isNull(), 1).otherwise(0))
    ).alias(c) for c in data.columns
    ]
)
subset.show()

+---+---------+---------+-----------+---------------+----+--------+-----------+-----+---------+---------+----------+--------------+----------+-------------------+-------+-----------+----------+---+-------+-------+-----+--------+----------+---+-----------+----------------+--------------+---------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+-----------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+----

In [16]:
sub = subset.toPandas()

In [17]:
sub = sub.melt(var_name="column", value_name="non_missing_count")

In [18]:
quantity_rows = data.count()

sub['non_missing_count'] = sub['non_missing_count'].astype(int).apply(lambda x: round(x / quantity_rows, 3))

In [19]:
sub.sort_values(by='non_missing_count', ascending=False)

,column,non_missing_count
1,member_id,1.000
49,next_pymnt_d,1.000
140,orig_projected_additional_accrued_interest,0.997
141,hardship_payoff_balance_amount,0.996
142,hardship_last_payment_amount,0.996
...,...,...
48,last_pymnt_amnt,0.000
110,tax_liens,0.000
128,hardship_flag,0.000
143,disbursement_method,0.000


In [ ]:
colunas = sub[sub['non_missing_count'] > 0.3]['column'].tolist()

In [26]:
data = data.select([
    f.col(c) for c in data.columns if c not in colunas
])

In [29]:
corr_ = {'columns': [], 'correlation': []}
columns_ = [c for c in data.columns if c != 'loan_status']
for c in columns_:
    try:
        corr_['correlation'].append(data.corr('loan_status', c))
        corr_['columns'].append(c)
    except:
        pass

In [34]:
sub = pd.DataFrame(corr_).sort_values(by='correlation', ascending=False)

In [46]:
colunas = sub[(sub['correlation'] > 0.01) | (sub['correlation'] < -0.01)]['columns'].tolist()
colunas.append('loan_status')

In [49]:
data = data.select(colunas)
len(data.columns)

30

Removendo nan

In [53]:
data.count()

1345309

In [55]:
data = data.dropna()

In [56]:
data.count()

1111188

## Criando modelo

In [74]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression

In [60]:
data.printSchema()

root
 |-- int_rate: double (nullable = true)
 |-- acc_open_past_24mths: double (nullable = true)
 |-- num_tl_op_past_12m: double (nullable = true)
 |-- num_actv_rev_tl: double (nullable = true)
 |-- num_rev_tl_bal_gt_0: double (nullable = true)
 |-- percent_bc_gt_75: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- num_actv_bc_tl: double (nullable = true)
 |-- num_op_rev_tl: double (nullable = true)
 |-- num_sats: double (nullable = true)
 |-- pub_rec_bankruptcies: double (nullable = true)
 |-- num_bc_sats: double (nullable = true)
 |-- pct_tl_nvr_dlq: double (nullable = true)
 |-- num_il_tl: double (nullable = true)
 |-- num_accts_ever_120_pd: double (nullable = true)
 |-- num_tl_90g_dpd_24m: double (nullable = true)
 |-- mo_sin_old_il_acct: double (nullable = true)
 |-- mths_since_recent_inq: double (nullable = true)
 |-- mo_sin_ol

In [80]:
data.select(['loan_status']).distinct().show()

+-----------+
|loan_status|
+-----------+
|          1|
|          0|
+-----------+



In [62]:
lista = [c for c in data.columns if c != 'loan_status']

In [63]:
assemble = VectorAssembler(
    inputCols=lista,
    outputCol="features"
)
data = assemble.transform(data)

In [64]:
train, test = data.randomSplit([0.7, 0.3], seed=42)

In [69]:
log = LogisticRegression(
    featuresCol='features',
    labelCol='loan_status',
)
model = log.fit(train)

In [70]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics

In [71]:
pred = model.transform(test)

In [89]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

pread = pred.select('loan_status', 'prediction').toPandas()

Exception ignored in: <function JavaModelWrapper.__del__ at 0x000001FE58766D40>
Traceback (most recent call last):
  File "c:\Users\mateu\Documents\Norton\Projetos GIT\livro-blueprints-de-aprendizado-de-maquina-e-cd-para-financas\venv\Lib\site-packages\pyspark\mllib\common.py", line 152, in __del__
    assert self._sc._gateway is not None
           ^^^^^^^^
AttributeError: 'BinaryClassificationMetrics' object has no attribute '_sc'


In [91]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, accuracy_score

print(confusion_matrix(pread['loan_status'], pread['prediction']))

[[259103   5005]
 [ 63339   5410]]


In [92]:
accuracy_score(pread['loan_status'], pread['prediction'])

0.7946745899890945

In [93]:
print(classification_report(pread['loan_status'], pread['prediction']))

              precision    recall  f1-score   support

           0       0.80      0.98      0.88    264108
           1       0.52      0.08      0.14     68749

    accuracy                           0.79    332857
   macro avg       0.66      0.53      0.51    332857
weighted avg       0.74      0.79      0.73    332857



In [94]:
from pyspark.ml.classification import DecisionTreeClassifier

In [95]:
tree = DecisionTreeClassifier(
    featuresCol='features',
    labelCol='loan_status'
)
model = tree.fit(train)
pred = model.transform(test)

In [96]:
pread = pred.select('loan_status', 'prediction').toPandas()

In [97]:
accuracy_score(pread['loan_status'], pread['prediction'])

0.7940917571209258

In [98]:
print(classification_report(pread['loan_status'], pread['prediction']))

              precision    recall  f1-score   support

           0       0.80      0.99      0.88    264108
           1       0.52      0.05      0.09     68749

    accuracy                           0.79    332857
   macro avg       0.66      0.52      0.49    332857
weighted avg       0.74      0.79      0.72    332857

